In [1]:
import numpy as np

In [20]:
np.random.seed(1)
H_test = np.random.rand(4, 4)
np.fill_diagonal(H_test, 0.0)
print(H_test)

[[0.00000000e+00 7.20324493e-01 1.14374817e-04 3.02332573e-01]
 [1.46755891e-01 0.00000000e+00 1.86260211e-01 3.45560727e-01]
 [3.96767474e-01 5.38816734e-01 0.00000000e+00 6.85219500e-01]
 [2.04452250e-01 8.78117436e-01 2.73875932e-02 0.00000000e+00]]


In [24]:
def candidate_star(H, beta: float):
    max_nodes = np.argmax(H, axis=1)
    stars = []
    for row_idx, max_idx in enumerate(max_nodes):
        w_max = H[row_idx, max_idx]
        star_indexes_y = np.argwhere(H[row_idx, :] >= w_max * beta)
        star_indexes = np.apply_along_axis(lambda x: np.hstack([row_idx, x]), 1, star_indexes_y)

        print(f"Star iteration: max value [{row_idx},{max_idx}] = {w_max}, with given threshold - {beta} {w_max * beta}")
        for star in star_indexes:
            print(f"\t\t-> star index{star}")

        stars.append(star_indexes)
    return stars

In [19]:
candidate_star(H_test, 0.5)

Star iteration: max value [0,1] = 0.7203244934421581, with given threshold - 0.5 0.36016224672107905
		-> star index[0 1]
Star iteration: max value [1,3] = 0.34556072704304774, with given threshold - 0.5 0.17278036352152387
		-> star index[1 2]
		-> star index[1 3]
Star iteration: max value [2,3] = 0.6852195003967595, with given threshold - 0.5 0.34260975019837975
		-> star index[2 0]
		-> star index[2 1]
		-> star index[2 3]
Star iteration: max value [3,1] = 0.8781174363909454, with given threshold - 0.5 0.4390587181954727
		-> star index[3 1]


In [29]:
def candidate_stars_2(H, beta: float):
    # key is the center node, values are candidates
    stars = {}

    for row_idx, row in enumerate(H):
        w_max = row.max()
        threshold = beta * w_max

        candidates = np.flatnonzero(row >= threshold)

        stars[row_idx] = candidates

    return stars

In [35]:
candidates = candidate_stars_2(H_test, 0.5)
candidates

{0: array([1]), 1: array([2, 3]), 2: array([0, 1, 3]), 3: array([1])}

In [40]:
def star_weight(H, center, candidates):
    nodes = np.hstack(([center], candidates)).astype(int)
    print(f"Nodes {nodes}")
    H_star = H[np.ix_(nodes, nodes)]
    return np.triu(H_star, k=1).sum()

In [41]:
scores = {
    center: star_weight(H_test, center, candidates)
    for center, candidates in candidates.items()
}

Nodes [0 1]
Nodes [1 2 3]
Nodes [2 0 1 3]
Nodes [3 1]


In [43]:
order = sorted(scores, key=scores.get, reverse=True)

In [42]:
scores

{0: np.float64(0.7203244934421581),
 1: np.float64(1.2170404388174783),
 2: np.float64(2.989021501747832),
 3: np.float64(0.8781174363909454)}

Extracting $H^{\star}_i$

In [44]:
candidates

{0: array([1]), 1: array([2, 3]), 2: array([0, 1, 3]), 3: array([1])}